### Setup from the 1st Demo

In [ ]:
import rootutils
path = rootutils.setup_root(search_from=__file__ if "__file__" in locals() else ".", indicator=".env", pythonpath=True)

import json
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain.messages import HumanMessage

# Connect to the local LM Studio server
llm = ChatOpenAI(
    base_url="http://localhost:1234/v1",
    api_key="lm-studio"
)

# Vulnerable source, contains sensitive data
@tool(name_or_callable="ReadContactsDB")
def read_local_contacts(query: str) -> str:
    """Useful for when you need to look up employee contacts, emails, or card details."""
    
    with open('../data/contacts.json', 'r', encoding='utf-8') as file:
        contacts = json.load(file)

    return f"""
        [CONFIDENTIAL DATABASE]
        {contacts}
    """

tools = [read_local_contacts]

# Create an agent
agent = create_agent(
    model=llm,
    tools=tools,
    debug=True
)

with open('../data/malicious_email.txt', 'r') as file:
    malicious_email = file.readlines()

# Launch the agent to process this letter
user_query = f"Analyze this email: {malicious_email}"
human_message = {
    "messages": [HumanMessage(user_query)]
}

agent_output = agent.invoke(human_message) # pyright: ignore[reportArgumentType]

last_message = agent_output['messages'][-1]
raw_text_response = last_message.content

print("\n--- RAW AGENT RESPONSE ---")
print(raw_text_response)

### Guardrail
Add and configure Misrosoft Presidio pipeline

In [ ]:
from src.middleware import secure_output_middleware

# Run output through MS Presidio pipeline
secure_response = secure_output_middleware(raw_text_response)
print("\n--- SECURE OUTPUT ---")
print(secure_response)

## Congfiguration

Custom ID pattern

In [ ]:
from src.middleware import custom_secure_output_middleware

# Run output through MS Presidio pipeline
secure_response = custom_secure_output_middleware(raw_text_response)
print("\n--- SECURE OUTPUT ---")
print(secure_response)

Custom mask

In [ ]:
from src.middleware import custom_secure_output_middleware_with_mask

secure_response = custom_secure_output_middleware_with_mask(raw_text_response)
print("\n--- SECURE OUTPUT ---")
print(secure_response)

Language

In [ ]:
from src.middleware import custom_secure_output_middleware_multiple_lang

secure_response = custom_secure_output_middleware_multiple_lang(raw_text_response)
print("\n--- SECURE OUTPUT ---")
print(secure_response)